In [1]:
import os
import numpy as np
import pandas as pd

# 1. LET PYTHON CREATE THE 'data' FOLDER FOR YOU
os.makedirs('data', exist_ok=True)

# Set seed for reproducibility
np.random.seed(2026)
n_patients = 1000

# 2. GENERATE CLINICAL ATTRIBUTES
patient_ids = [f"PT-{i:04d}" for i in range(1, n_patients + 1)]
age = np.random.randint(18, 85, size=n_patients)
bmi = np.random.normal(loc=26.5, scale=5.0, size=n_patients).round(1)
# Choosing random dosages between 200mg, 400mg, 600mg, and 800mg
drug_dosage_mg = np.random.choice([200, 400, 600, 800], size=n_patients)

# 3. EMBED HIDDEN BIOLOGICAL INTERACTION RULES
# High dosage + high BMI strictly increases adverse events
toxicity_score = (0.005 * drug_dosage_mg) + (0.12 * bmi) + (0.01 * age) - 5.0
prob_adverse_event = 1 / (1 + np.exp(-toxicity_score))
adverse_event = np.random.binomial(n=1, p=prob_adverse_event)

# 4. CREATE THE DATAFRAME
df_health = pd.DataFrame({
    'PatientID': patient_ids,
    'Age': age,
    'BMI': bmi,
    'Dosage_MG': drug_dosage_mg,
    'AdverseEvent': adverse_event
})

# Simulate real-world human error: missing BMI logs for elderly patients
df_health.loc[df_health['Age'] > 75, 'BMI'] = np.nan

# 5. SAVE DIRECTLY INTO YOUR NEW FOLDER
df_health.to_csv('data/raw_clinical_trials.csv', index=False)
print("Setup complete! Your 'data' folder and 'raw_clinical_trials.csv' file have been generated.")


Setup complete! Your 'data' folder and 'raw_clinical_trials.csv' file have been generated.


In [2]:
import pandas as pd
import numpy as np

# 1. Load your newly generated raw CSV file
df = pd.read_csv('data/raw_clinical_trials.csv')

# 2. View the first 5 rows to see what the data looks like
print("--- FIRST 5 ROWS ---")
print(df.head(), "\n")

# 3. Check for missing values (This reveals the data friction)
print("--- MISSING VALUES PER COLUMN ---")
print(df.isnull().sum(), "\n")

# 4. Handle the missing data: Impute the missing BMI values
# We calculate the median BMI of the dataset to fill in the missing logs
median_bmi = df['BMI'].median()
df['BMI'] = df['BMI'].fillna(median_bmi)

print(f"Successfully cleaned data! Missing BMIs filled with median value: {median_bmi}")
print(f"Remaining missing values in dataset: {df['BMI'].isnull().sum()}")


--- FIRST 5 ROWS ---
  PatientID  Age   BMI  Dosage_MG  AdverseEvent
0   PT-0001   19  23.4        600             1
1   PT-0002   24  25.7        800             1
2   PT-0003   44  19.4        200             1
3   PT-0004   74  28.6        400             0
4   PT-0005   47  27.1        600             1 

--- MISSING VALUES PER COLUMN ---
PatientID         0
Age               0
BMI             129
Dosage_MG         0
AdverseEvent      0
dtype: int64 

Successfully cleaned data! Missing BMIs filled with median value: 26.5
Remaining missing values in dataset: 0


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# 1. Pivot Table: Calculate the percentage of patients experiencing an adverse event by Dosage
print("--- RISK BY DOSAGE LEVEL ---")
dosage_pivot = df.groupby('Dosage_MG')['AdverseEvent'].mean() * 100
for dosage, risk in dosage_pivot.items():
    print(f"Dosage {dosage}mg: {risk:.1f}% of patients had an adverse event")
print("\n")

# 2. Train a Logistic Regression model to find mathematical weights
X = df[['Age', 'BMI', 'Dosage_MG']]
y = df['AdverseEvent']
model = LogisticRegression().fit(X, y)

# 3. Print the impact coefficients
print("--- RISK MULTIPLIERS (MODEL COEFFICIENTS) ---")
features = X.columns
coefficients = model.coef_[0]
for feature, coef in zip(features, coefficients):
    print(f"{feature}: {coef:.4f}")


--- RISK BY DOSAGE LEVEL ---
Dosage 200mg: 42.6% of patients had an adverse event
Dosage 400mg: 61.3% of patients had an adverse event
Dosage 600mg: 78.6% of patients had an adverse event
Dosage 800mg: 92.3% of patients had an adverse event


--- RISK MULTIPLIERS (MODEL COEFFICIENTS) ---
Age: 0.0083
BMI: 0.1239
Dosage_MG: 0.0047


# Portfolio Project Phase 1: Clinical Trial Safety Optimization Framework
**Author:** Isa Siddique  
**Industry:** Healthcare / Pharmaceuticals  

---

## Executive Summary
This analysis evaluated clinical trial monitoring logs for 1,000 patients to identify systemic biological triggers causing adverse reactions to a trial drug phase. The overarching goal is to define actionable clinical guidelines that mitigate patient risk and optimize future rollout safety.

## Data Engineering & Cleaning Notes
* **Data Friction Solved:** Uncovered an operational data gap where patient BMI logs were missing for 129 elderly patients over the age of 75.
* **Methodology:** Rather than dropping these records (which would introduce extreme demographic bias against elderly metrics), a median imputation strategy ($BMI = 26.5$) was deployed to fully preserve sample integrity.

## Core Analytics Insights & Mathematical Proof
1. **Dosage Escalation Threshold:** Adverse drug events increase non-linearly with dosage sizes. Moving from 200mg to 800mg spikes the patient complication rate from **42.6% to an alarming 92.3%**.
2. **Physiological Risk Multipliers:** Logistic Regression modeling isolated the true drivers of systemic toxicity:
   * **Age ($\beta = 0.0083$):** Acts as a mild, progressive baseline risk multiplier.
   * **Dosage ($\beta = 0.0047$):** Increases danger progressively across all cohorts.
   * **BMI ($\beta = 0.1239$):** Emerging as the primary risk factor. A patient's body mass index has a dramatic compounding impact on overall drug toxicity.

---

## Strategic Proposal & Clinical Guidelines

Based directly on the coefficients and risk percentages uncovered above, the following protocol modifications are proposed for immediate integration into the Electronic Health Record (EHR) systems for Phase 2:

### 1. Hard-Capped Patient Stratification Mandate
Because BMI ($\beta = 0.1239$) is the most violent risk accelerator, patients must be strictly stratified prior to dosing.
* **Guideline:** Implement a hard-coded software block preventing clinicians from escalating any patient with a documented BMI > 28 above a baseline **400mg limit** (where total cohort risk remains manageable).

### 2. Mandatory Step-Down Titration Protocol
The data shows that 800mg is highly toxic across the board (92.3% event rate).
* **Guideline:** Eliminate the 800mg deployment group entirely from standard treatment pathways. Replace it with a maximum cap of 600mg, alongside a mandatory step-down titration window where a patient is monitored for 72 hours post-dose before any subsequent administration.

### 3. Automated EHR Alerts for Elderly Cohorts
Because Age holds a steady positive risk coefficient, older patients require tighter guardrails.
* **Guideline:** Automatically trigger a "High-Risk Monitoring Flag" in the patient database if an individual is over the age of 65 and assigned a dose greater than 200mg, requiring daily telemetry updates.
